# Installs

<li>pip install mne<li>
<li>pip install pandas<li>
<li>pip install numpy<li>
<li>pip install torch<li>
<li>pip install torcheeg<li>

In [ ]:

import os
import random
import time
from mne.datasets import eegbci
import mne
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from matplotlib import pyplot as plt
from sklearn import metrics
from torcheeg import transforms
from collections import Counter
from torcheeg.model_selection import KFold
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
edffiles=r"C:\Users\johan\Documents\Documents\School\advanced machine learning\PROJECT\BCI"
subjects_id=np.arange(1,5)
runs_id=np.arange(4,10)
paths=eegbci.load_data(subjects=subjects_id, runs=runs_id,path=edffiles, update_path=True)
paths

# Check cuda is available


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
edffile=r'C:/Users/johan/Documents/Documents/School/advanced machine learning/PROJECT/BCI/MNE-eegbci-data/files/eegmmidb/1.0.0/S003/S003R03.edf'


raw=mne.io.read_raw_edf(edffile)
raw.info

In [ ]:
print("duration: ",raw.annotations.duration)
print("count: ",raw.annotations.count())
print("onset:  ",raw.annotations.onset)
print("channel names: ",raw.ch_names)
raw.annotations.to_data_frame()


In [ ]:
events_=mne.events_from_annotations(raw)
counter=0
for i in events_[0]:
    print(f"number is {counter +1}: ",i)
    counter +=1

In [ ]:
def get_X_y(path: str, movement: str )-> tuple[list,list]:
    raw_data=mne.io.read_raw_edf(path)
    durations=raw.annotations.duration
    onset=raw.annotations.onset
    events_=mne.events_from_annotations(raw_data)
    tasks={}
    X=[]
    y=[]
    counter=0
    match movement:
        case 'lr':
            tasks={1:0, 2:1, 3:2}
            
        case 'ilr':
            tasks={1:0, 2:3, 3:4}
        case 'ff':
            tasks={1:0, 2:5, 3:6}
        case 'iff':
            tasks={1:0, 2:7, 3:8}
        case _:

            raise ValueError(f"movement: {movement} is not supported. Chose either lr, ilr, ff or iff")
    
    for e,o,d in zip(events_[0],onset,durations):
        X.append(raw_data.get_data(start=int ( o * 160), stop= int (( d * 160) + ( o * 160))))
        y.append(e[2])
        counter+=1

    y=list(map(lambda x: tasks[x], y))
    return X, y



In [ ]:
edffiles=r"C:\Users\johan\Documents\Documents\School\advanced machine learning\PROJECT\BCI"
lr=[3,7,11]
ilr=[4,8,12]
ff=[5,9,13]
iff=[6,10,14]
movements=[lr,ilr,ff,iff]
mov_str=['lr','ilr','ff','iff']
X=[]
y=[]
for m,sr in zip(movements,mov_str):
    paths=eegbci.load_data(subjects=[1], runs=m,path=edffiles, update_path=True)
    for path in paths:
        Xs,ys=get_X_y(path,sr)
        X.extend(Xs)
        y.extend(ys)

len(y)

### Normalize

In [ ]:
for i in range (len(X)):
    mean=np.mean(X[i],axis=1,keepdims=True) 
    std=np.std(X[i],axis=1,keepdims=True)
    X[i]=(X[i]-mean)/std




In [ ]:
raw.plot(show_options=True)
first_ch=raw[0][0]
first_ch
plt.plot(first_ch[0])
plt.show()


In [ ]:
data=raw.get_data()
ch_1=data[0]

hs=160
n=np.arange(1, 3201)
n=n/hs # set seconds
plt.plot(n,ch_1[:3200]) # first 20 seconds = 160 X 20
plt.title("not normalized")
plt.show()

plt.plot(X[0][0][:3200])
plt.title("normalized data")
plt.show()

In [ ]:
#treatment of unequal sized tensors

counter=1
zeros=np.zeros((64,16)) #padding with zeros so all segments are of same size
for i in range(len(X)):
    if i%2 ==0:
        X[i]=X[i][:,:656]
        mean=mean+X[i]
        counter+=1
    
mean=mean/counter
print(mean)
print(X[1].shape)
print(X[0].shape)


In [ ]:
plt.plot(mean.T)
plt.title("Mean base state of 64 channels")
plt.show()
plt.plot(mean[1])
plt.title("Mean base state of one channel")
plt.show()
plt.plot(X[0].T)
plt.title("Base state")
plt.show()


In [ ]:

X_t= torch.tensor(X, dtype=torch.float32)
y_t= torch.tensor(y, dtype=torch.long)



X_t_s = X_t.unsqueeze(1)
print(X_t_s.shape)
n=len(X_t_s)
training_end=int(n*0.7)
validation_end=int(n*0.85)

X_train_1D, y_train=X_t[:training_end], y_t[:training_end]
X_val_1D, y_val=X_t[training_end:validation_end], y_t[training_end:validation_end]
X_test_1D,y_test=X_t[validation_end:],y_t[validation_end:]

X_train_2D=X_t_s[:training_end]
X_val_2D=X_t_s[training_end:validation_end]
X_test_2D=X_t_s[validation_end:]

print(X_train_2D.shape)
print(X_val_2D.shape)
print(X_test_2D.shape)



In [ ]:
#1D dataloaders
training_dataset_1D=TensorDataset(X_train_1D,y_train)
val_dataset_1D=TensorDataset(X_val_1D,y_val)
test_dataset_1D=TensorDataset(X_test_1D,y_test)
train_loader_1D = DataLoader(training_dataset_1D, batch_size=30, shuffle=True,generator=torch.Generator().manual_seed(42))
val_loader_1D = DataLoader(val_dataset_1D, batch_size=30, shuffle=False,generator=torch.Generator().manual_seed(42))
test_loader_1D = DataLoader(test_dataset_1D, batch_size=30, shuffle=False,generator=torch.Generator().manual_seed(42))

#2D dataloaders
training_dataset_2D=TensorDataset(X_train_2D,y_train)
val_dataset_2D=TensorDataset(X_val_2D,y_val)
test_dataset_2D=TensorDataset(X_test_2D,y_test)
train_loader_2D = DataLoader(training_dataset_2D, batch_size=30, shuffle=True,generator=torch.Generator().manual_seed(42))
val_loader_2D = DataLoader(val_dataset_2D, batch_size=30, shuffle=False,generator=torch.Generator().manual_seed(42))
test_loader_2D = DataLoader(test_dataset_2D, batch_size=30, shuffle=False,generator=torch.Generator().manual_seed(42))



In [ ]:

class eggNetWork(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv2d= nn.Sequential(
            #block1
            nn.Conv2d(1,16,(1,40),padding=10),
            nn.BatchNorm2d(16),
            nn.ELU(),
            

            #block2
            nn.Conv2d(16,32,(64,3),padding=10),
            nn.BatchNorm2d(32),
            nn.ELU(),
           

            #block3
            nn.Conv2d(32,64,(20,20),padding=10),
            nn.ELU(),
            nn.BatchNorm2d(64),
            

            nn.AdaptiveAvgPool2d(2),
            nn.Flatten()  
        )

        self.conv1d= nn.Sequential(
            #block1
            nn.Conv1d(64,128,40,padding=40),
            nn.MaxPool1d(2,stride=2),
            nn.ELU(),
            nn.BatchNorm1d(128),

            #block2
            nn.Conv1d(128,256,10,padding=10),
            nn.MaxPool1d(10,stride=2),
            nn.ELU(),
            nn.BatchNorm1d(256),

             #block3
            nn.Conv1d(256,128,10,padding=10),
            nn.MaxPool1d(10,stride=2),
            nn.ELU(),
            nn.BatchNorm1d(128),

            #block4
            nn.Conv1d(128,40,10,padding=10),
            nn.MaxPool1d(10,stride=2),
            nn.ELU(),
            nn.BatchNorm1d(40),

            #block5
            nn.Conv1d(40,20,20,padding=20),
            nn.ELU(),
            nn.BatchNorm1d(20), 

            #block6
            nn.Conv1d(20,10,20,padding=20),
            nn.ELU(),
            nn.BatchNorm1d(10), 

            #block7
            nn.Conv1d(10,5,20,padding=20),
            nn.ELU(),
            nn.BatchNorm1d(5),


            nn.AdaptiveAvgPool1d(64),
            nn.Flatten()
        )

        self.fc=nn.Sequential(
            nn.Linear(320+64*2*2, 9),
            
            

        )

        

    def forward(self, x1d,x2d):
        conv1d=self.conv1d(x1d)
        conv2d=self.conv2d(x2d)
        x=torch.cat((conv1d,conv2d),dim=1)
        out=self.fc(x)
        return out


In [ ]:
module=eggNetWork()

optimizer = torch.optim.SGD(module.parameters(), lr=0.005, momentum=0.9)


In [ ]:
def criterion(y):
    num_classes = 9

    class_counts = torch.bincount(y, minlength=num_classes)
    class_weights = 1.0 / class_counts.float()
    class_weights = class_weights / class_weights.sum() * num_classes

    class_weights = class_weights.to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    return criterion

In [ ]:
def training_and_evaluation(module ,device,opt ,train_loader_1D,train_loader_2D,val_loader_1D,val_loader_2D):
    
    

    module.train()
    total_loss=0
    val_loss=0
    for (X1,y1),(X2,y2) in zip(train_loader_1D,train_loader_2D):
        criterion=criterion(y2)

        X1, X2, y1, y2=X1.to(device), X2.to(device), y1.to(device), y2.to(device)
        optimizer.zero_grad() #rest the greadient
        
        pred=module(X1,X2) 
        loss=criterion(pred,y1)
        loss.backward()
        opt.step()
        avrg_batch_loss=loss.item()
        total_loss+=loss.item()
    
    print(f"total_loss: {total_loss}")

    module.eval()
    with torch.no_grad():
        print("evaluation")
        counter=0
        all_pred=[]
        for (X1,y1),(X2,y2) in zip(val_loader_1D,val_loader_2D):
            X1, X2, y1, y2=X1.to(device), X2.to(device), y1.to(device), y2.to(device)
            pred=module(X1,X2)
            
            loss=criterion(pred,y1)
            
            val_loss+=loss.item()
            counter+=1
            predictions=pred.argmax(axis=1)
            all_pred.extend([int(v) for v in predictions])
            
        print(f"val_loss: {val_loss}")
        print(f"prediction distribution:{Counter(all_pred)}")
        print(f"logits: {pred}")


    print("Finnished training")
        
        

In [ ]:
def testing(module,device, test_loader_1D,test_loader_2D):
    module.eval()
    total_loss=0
    correct=0
    total=0
    with torch.no_grad():
        for (X1,y1),(X2,y2) in zip(test_loader_1D,test_loader_2D):
            X1, X2, y1, y2=X1.to(device), X2.to(device), y1.to(device), y2.to(device)
            pred=module(X1,X2)
            loss=criterion(pred,y1)
            total_loss+=loss.item()
            _,indices=torch.max(pred,1)
            correct+=(indices==y1).sum().item()
            total+=len(y1)
        print(100*(correct/total))

    

In [ ]:



training_and_evaluation(module=module,device=device,opt=optimizer ,train_loader_1D=train_loader_1D,train_loader_2D=train_loader_2D,val_loader_1D=val_loader_1D,val_loader_2D=val_loader_2D)

In [ ]:
testing(module,device=device,test_loader_1D=test_loader_1D,test_loader_2D=test_loader_2D)